# QARP Factor-Based Selection Engine

## Summary

| | |
| :--- | :--- |
| **Idea** | Rank IT-services companies each year by a score combining revenue growth, ROIC and Piotroski F-score, divided by relative price-to-sales, and hold the top five. |
| **Verdict** | Inconclusive: the result is not credible |
| **Result** | The backtest reports -76.8% from 2016 against +340% for SPY (CAGR -12.9%). |
| **Trust** | Very low. A basket that held CDW, DELL, SAIC and EPAM for most of a decade losing three quarters of its value is almost certainly a fault in the backtest loop, not a finding. It also ran on MacroTrends fundamentals that covered few tickers, used today's industry valuation medians for past dates, and chose its universe with hindsight. The factor ideas themselves were never fairly tested. |
| **Date** | 2026-07-22 |
| **Code status** | Frozen: written before the current modules and no longer runs. Read it as a report. |

*Summary added 2026-10-13 by Claude from this notebook's own commentary and stored results. The verdict is a draft for Ben to confirm or correct. Worth redoing on the EDGAR data and FundamentalIndicators, which did not exist when this was written.*

# QARP Factor-Based Selection Engine (Institutional Grade)
This notebook implements a modular "Point-in-Time" (PIT) factor engine. We separate data loading, atomic factor extraction, and portfolio selection to ensure we can stress-test each thesis independently.

---

## PHASE 1: DATA PIPELINE & ATOMIC LIBRARIES
In this phase, we load the global financial datasets and define the "Atomic" functions that extract features at a specific point in time.

### Step 1: Global Data Hub
Load fundamental and price data from the local repository.

In [142]:
# --- PHASE 1, STEP 1: GLOBAL DATA HUB ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import yfinance as yf
from datetime import datetime, timedelta
import os

print("--- Loading Global Financial Datasets ---")

# 1. Ticker Metadata & Industry Medians
tickers_meta = pd.read_csv('MT/mt-tickers.csv').set_index('ticker')
ind_medians = pd.read_csv('MT/mt-industries.csv').set_index('zacks_x_ind_desc')

# 2. Fundamental Statements
inc_raw = pd.read_csv('MT/mt-income-statement-quarterly.csv')
bal_raw = pd.read_csv('MT/mt-balance-sheet-quarterly.csv')
cf_raw = pd.read_csv('MT/mt-cash-flow-statement-quarterly.csv')

# 3. Price Data (Standardized Local Source)
# Logic: We load from the local CSV.
price_file = 'YF/yf-prices.csv'
price_df = pd.read_csv(price_file)
price_df['date'] = pd.to_datetime(price_df['date'])

for df in [inc_raw, bal_raw, cf_raw]:
    df['Quarter'] = pd.to_datetime(df['Quarter'])

it_services_universe = tickers_meta[tickers_meta['zacks_x_ind_desc'] == 'Information Technology Services'].index.unique().tolist()
print(f"Phase 1.1 Complete. Universe: {len(it_services_universe)} tickers (incl. local SPY benchmark).")


--- Loading Global Financial Datasets ---
Phase 1.1 Complete. Universe: 58 tickers (incl. local SPY benchmark).


### PHASE 1, Step 2: Atomic PIT Factor Library
This library ensures every calculation follows a consistent pattern: `factor(ticker, date)`. We strictly enforce a 60-day reporting lag to prevent look-ahead bias.

In [151]:
# --- PHASE 1, STEP 2: ATOMIC PIT LIBRARY ---

def get_ttm_at_date(ticker, df_raw, metric, target_date):
    """Retrieves TTM sum with a 60-day lag."""
    lag_date = pd.to_datetime(target_date) - timedelta(days=60)
    subset = df_raw[(df_raw['ticker'] == ticker) & (df_raw['Quarter'] < lag_date)].sort_values('Quarter')
    if len(subset) < 4: return np.nan
    if (subset['Quarter'].iloc[-1] - subset['Quarter'].iloc[-4]).days > 550: return np.nan
    return pd.to_numeric(subset[metric].tail(4), errors='coerce').sum()

def get_latest_at_date(ticker, df_raw, metric, target_date):
    """Retrieves the latest available single-quarter value."""
    lag_date = pd.to_datetime(target_date) - timedelta(days=60)
    subset = df_raw[(df_raw['ticker'] == ticker) & (df_raw['Quarter'] < lag_date)].sort_values('Quarter')
    if subset.empty: return np.nan
    return pd.to_numeric(subset[metric].iloc[-1], errors='coerce')

def get_price_at_date(ticker, target_date, adjusted=False):
    """
    Standardized price lookup. 
    Use adjusted=False (NOMINAL CLOSE) for Market Cap and Valuation.
    Use adjusted=True (ADJ CLOSE) for returns and performance analysis.
    """
    p_subset = price_df[(price_df['ticker'] == ticker) & (price_df['date'] <= pd.to_datetime(target_date))]
    if p_subset.empty: return np.nan
    
    # Logic: Valuation vs Returns
    col = 'adj_close' if adjusted else 'close'
    return p_subset[col].iloc[-1]


# --- CORE ATOMIC FUNCTIONS ---

def ttm_revenue(ticker, date):
    return get_ttm_at_date(ticker, inc_raw, 'Revenue', date)

def market_cap(ticker, date):
    # CRITICAL: Always use nominal 'close' to match nominal MacroTrends shares/revenue
    price = get_price_at_date(ticker, date, adjusted=False)
    shares = get_latest_at_date(ticker, inc_raw, 'Shares Outstanding', date)
    return price * shares if not (np.isnan(price) or np.isnan(shares)) else np.nan

def ttm_roic(ticker, date):
    op_inc = get_ttm_at_date(ticker, inc_raw, 'Operating Income', date)
    lag_date = pd.to_datetime(date) - timedelta(days=60)
    b_subset = bal_raw[(bal_raw['ticker'] == ticker) & (bal_raw['Quarter'] < lag_date)].sort_values('Quarter')
    if len(b_subset) < 4: return np.nan
    assets = pd.to_numeric(b_subset['Total Assets'], errors='coerce')
    cash = pd.to_numeric(b_subset['Cash On Hand'], errors='coerce').fillna(0)
    inv_cap = (assets - cash).tail(4).mean()
    return (op_inc / inv_cap * 100) if (inv_cap and inv_cap > 0) else np.nan

print("Phase 1.2 Complete. Atomic Library Loaded.")

Phase 1.2 Complete. Atomic Library Loaded.


### PHASE 1, Step 3: Piotroski F-Score (Survival Score)
The F-Score measures 9 factors of financial strength. We use this as an atomic survival signal to filter out "junk" companies.

In [152]:
# --- PHASE 1, STEP 3: PIOTROSKI F-SCORE LIBRARY ---

def f_roa(ticker, date):
    ni = get_ttm_at_date(ticker, inc_raw, 'Net Income', date)
    assets = get_latest_at_date(ticker, bal_raw, 'Total Assets', date)
    return 1 if (assets and assets > 0 and (ni / assets) > 0) else 0

def f_cfo(ticker, date):
    cfo = get_ttm_at_date(ticker, cf_raw, 'Cash Flow From Operating Activities', date)
    return 1 if cfo > 0 else 0

def f_delta_roa(ticker, date):
    p_year = pd.to_datetime(date) - timedelta(days=365)
    roa_now = get_ttm_at_date(ticker, inc_raw, 'Net Income', date) / get_latest_at_date(ticker, bal_raw, 'Total Assets', date)
    roa_prev = get_ttm_at_date(ticker, inc_raw, 'Net Income', p_year) / get_latest_at_date(ticker, bal_raw, 'Total Assets', p_year)
    return 1 if roa_now > roa_prev else 0

def f_accrual(ticker, date):
    cfo = get_ttm_at_date(ticker, cf_raw, 'Cash Flow From Operating Activities', date)
    ni = get_ttm_at_date(ticker, inc_raw, 'Net Income', date)
    assets = get_latest_at_date(ticker, bal_raw, 'Total Assets', date)
    return 1 if (assets and assets > 0 and (cfo / assets) > (ni / assets)) else 0

def piotroski_f_score(ticker, date):
    # Consolidating the core strength factors
    factors = [f_roa(ticker, date), f_cfo(ticker, date), f_delta_roa(ticker, date), f_accrual(ticker, date)]
    return sum(factors) * 2.25 # Scalar to maintain 0-9 scale impact

print("Phase 1.3 Complete. F-Score Library Loaded.")

Phase 1.3 Complete. F-Score Library Loaded.


### PHASE 1, Step 4: Growth Momentum (6-6-35)
This is our primary filter for finding high-conviction compounders. It requires 6 quarters of consecutive growth and 35% cumulative expansion.

In [153]:
# --- PHASE 1, STEP 4: MOMENTUM & VALUATION ANCHORS (ATOMIC) ---

def get_cumulative_growth(ticker, date, quarters=6):
    """Calculates internal revenue growth over the specified number of quarters."""
    series = get_quarterly_series(ticker, inc_raw, 'Revenue', date, quarters=quarters+1)
    if series is None or series.isnull().any() or series.iloc[0] <= 0: return np.nan
    return ((series.iloc[-1] / series.iloc[0]) - 1) * 100

def get_historical_ps_median(ticker, date, years=3):
    """Retrieves the ticker's own median P/S over the historical window."""
    p_dates = [pd.to_datetime(date) - timedelta(days=365*i) for i in range(years + 1)]
    past_ps = []
    for d in p_dates:
        rev = ttm_revenue(ticker, d)
        mkt = market_cap(ticker, d)
        if rev and rev > 0 and mkt:
            past_ps.append(mkt / rev)
    return np.nanmedian(past_ps) if past_ps else np.nan

print("Phase 1.4 Complete. Atomic Primitives for Growth & Valuation Loaded.")

Phase 1.4 Complete. Atomic Primitives for Growth & Valuation Loaded.


## PHASE 2: UNIFIED FACTOR MATRIX
We now apply our atomic libraries to the entire universe to create a master data table for a specific point in time. This matrix allows for cross-sectional ranking.

In [154]:
# --- PHASE 2: ATOMIC FACTOR MATRIX ---
# Populating the master table with raw primitives and atomic financial ratios.
# Note: Ratios like Rel_PS are included here as they are objective financial calculations.

snapshot_date = datetime.now().date() 

master_factors = pd.DataFrame(index=it_services_universe)
master_factors.index.name = 'Ticker'

print(f"--- Generating Phase 2 Atomic Matrix for {snapshot_date} ---")

# 1. Fundamental Primitives
master_factors['Revenue_TTM'] = [ttm_revenue(t, snapshot_date) for t in master_factors.index]
master_factors['Market_Cap'] = [market_cap(t, snapshot_date) for t in master_factors.index]

# 2. Quality & Strength Factors
master_factors['ROIC_%'] = [ttm_roic(t, snapshot_date) for t in master_factors.index]
master_factors['F_Score_Factor'] = [piotroski_f_score(t, snapshot_date) for t in master_factors.index]

# 3. Momentum Primitive
master_factors['Growth_6Q_%'] = [get_cumulative_growth(t, snapshot_date) for t in master_factors.index]

# 4. Atomic Valuation Ratios
# Current P/S
master_factors['PS_Current'] = master_factors['Market_Cap'] / master_factors['Revenue_TTM'].replace(0, np.nan)

# Relative to Industry Median
master_factors['Rel_PS_Ind'] = master_factors.apply(
    lambda row: row['PS_Current'] / ind_medians.loc[tickers_meta.loc[row.name, 'zacks_x_ind_desc'], 'median_ps'] 
    if row.name in tickers_meta.index else np.nan, axis=1
)

# Relative to Own History
master_factors['Rel_PS_Hist'] = master_factors.apply(
    lambda row: row['PS_Current'] / get_historical_ps_median(row.name, snapshot_date) 
    if row['PS_Current'] > 0 else np.nan, axis=1
)

master_factors = master_factors.dropna(subset=['Revenue_TTM', 'Market_Cap'])

print("Phase 2 Complete. Primitives and Ratios populated.")
display(master_factors.head())

--- Generating Phase 2 Atomic Matrix for 2026-07-21 ---
Phase 2 Complete. Primitives and Ratios populated.


,Revenue_TTM,Market_Cap,ROIC_%,F_Score_Factor,Growth_6Q_%,PS_Current,Rel_PS_Ind,Rel_PS_Hist
Ticker,,,,,,,,
NOW,13960.0000,106978.066500,10.531930,6.75,34.787272,7.663185,2.941722,0.477990
ROP,8115.0000,37145.027979,6.718849,9.00,18.740791,4.577329,1.757132,0.513501
DELL,134002.0000,262400.000000,12.089741,9.00,79.931051,1.958180,0.751700,2.052002
CSGP,3411.9999,12014.280190,-0.316015,6.75,29.511984,3.521184,1.351702,0.288582
CDW,22904.8000,16881.620079,10.985681,6.75,2.958344,0.737034,0.282931,0.658828


## PHASE 3: THE OPPORTUNITY MATRIX (SECTOR VIEW)
We rank the universe based on our integrated QARP logic:
$$ \text{Stack Score} = \frac{(0.5 \times \text{Growth}) + (0.3 \times \text{ROIC}) + (5 \times \text{F-Score})}{\text{Dual-Anchor P/S}} $$

In [155]:
# --- PHASE 3: OPPORTUNITY MATRIX (STRATEGY DERIVATION & FILTERING) ---
# Here we implement our opinionated strategy: filtering, derivation, and weighting.

def build_opportunity_matrix(factors_df):
    """
    Transforms atomic factors into a filtered opportunity matrix.
    Treats all inputs (Revenue, MC, ROIC) as atomic factors.
    Filters represent the 'Institutional Survival Floor' of the thesis.
    """
    df = factors_df.copy()
    
    # 1. Survival Floors (Institutional Quality Filters)
    # These are part of the thesis logic, not external parameters.
    SURVIVAL_REV_FLOOR = 500  # $500M TTM Revenue
    SURVIVAL_MKT_FLOOR = 1000 # $1B Market Cap
    
    df = df[(df['Revenue_TTM'] >= SURVIVAL_REV_FLOOR) & 
            (df['Market_Cap'] >= SURVIVAL_MKT_FLOOR)]
    
    # 2. Derived Valuation Scores (Atomic Anchor Averaging)
    df['Combined_Valuation'] = (df['Rel_PS_Ind'] + df['Rel_PS_Hist']) / 2
    
    # 3. Decision Formula: The Strategy Stack
    # Weighting: 40% Growth, 40% ROIC, 20% F-Score Equivalent
    df['Stack_Score'] = (
        (df['Growth_6Q_%'] * 0.4) + 
        (df['ROIC_%'] * 0.4) + 
        (df['F_Score_Factor'] * 5)
    ) / df['Combined_Valuation']
    
    return df.sort_values('Stack_Score', ascending=False)

# Implementation for Today's Dashboard
opportunity_matrix = build_opportunity_matrix(master_factors)

print(f"--- Top Ranked Candidates (Phase 3 Opportunity Logic) ---")
cols_to_show = ['Revenue_TTM', 'ROIC_%', 'F_Score_Factor', 'Growth_6Q_%', 'Combined_Valuation', 'Stack_Score']
display(opportunity_matrix[cols_to_show].head(20))

--- Top Ranked Candidates (Phase 3 Opportunity Logic) ---


,Revenue_TTM,ROIC_%,F_Score_Factor,Growth_6Q_%,Combined_Valuation,Stack_Score
Ticker,,,,,,
EPAM,5555.4250,14.779024,6.75,19.916798,0.358405,132.889827
DXC,12644.0000,4.083676,6.75,-3.424869,0.294513,115.490679
SAIC,7291.0000,11.064399,9.00,-3.542510,0.564381,85.064459
UPST,1138.7001,2.077176,4.50,90.091279,0.702242,84.539717
CDW,22904.8000,10.985681,6.75,2.958344,0.470879,83.519522
TBLA,1950.9420,8.369632,9.00,7.709486,0.651008,79.003049
DOX,4622.5730,11.997128,6.75,-7.271486,0.536530,66.427372
WIX,2060.5640,-9.995641,4.50,21.700890,0.444394,61.166606
DELL,134002.0000,12.089741,9.00,79.931051,1.401851,58.357349


## PHASE 4: FACTOR ATTRIBUTION (THE EVIDENCE)
We look back to 2016 to see if our atomic factors actually correlated with 8-year performance. This provides the statistical validation for our strategy pivot.

In [156]:
# --- PHASE 4: STATISTICAL ATTRIBUTION (2016-2024) ---
# Testing the predictive power of our Atomic Factors over an 8-year horizon.

def run_phase4_attribution(tickers, start='2016-01-01', end='2024-01-01'):
    data = []
    # Using local truth for price returns
    all_prices = price_df.pivot(index='date', columns='ticker', values='adj_close')
    
    print(f"--- Running 8-Year Attribution analysis for {len(tickers)} tickers ---")
    
    for t in tickers:
        try:
            # 1. Baseline 2016 Atomic Factors
            # We calculate these for a point-in-time (2016)
            rev = ttm_revenue(t, start)
            mkt = market_cap(t, start)
            if not rev or rev <= 0 or not mkt: continue
            
            roic = ttm_roic(t, start)
            growth = get_cumulative_growth(t, start)
            f_score = piotroski_f_score(t, start)
            
            # Dual Anchor Valuation
            ps_curr = mkt / rev
            rel_ind = ps_curr / ind_medians.loc[tickers_meta.loc[t, 'zacks_x_ind_desc'], 'median_ps']
            rel_hist = ps_curr / get_historical_ps_median(t, start)
            combined_val = (rel_ind + rel_hist) / 2
            
            # 2. Total Return Calculation (Local Data)
            sub = all_prices[t].dropna()
            if sub.empty: continue
            
            p_start = sub.loc[sub.index >= pd.to_datetime(start)]
            p_end = sub.loc[sub.index <= pd.to_datetime(end)]
            
            if p_start.empty or p_end.empty: continue
            
            ret = (p_end.iloc[-1] / p_start.iloc[0]) - 1
            
            data.append({
                'Ticker': t, 'Return': ret, 'ROIC': roic, 
                'Growth': growth, 'F_Score': f_score, 
                'Valuation': combined_val, 'Rev_Size': np.log(rev)
            })
        except Exception as e:
            continue
        
    attr_df = pd.DataFrame(data).dropna()
    print("--- Phase 4: Correlation with 8-Year Total Return ---")
    corr = attr_df.select_dtypes(include=[np.number]).corr()['Return'].sort_values(ascending=False)
    print(corr)
    return attr_df

# attr_results = run_phase4_attribution(it_services_universe)

## PHASE 5: THE STRATEGY PIVOT & BACKTEST
Based on our Phase 4 findings, we prioritize Growth and Survival. We now run the full modular backtest from 2016 to verify if this "Atomic QARP" strategy consistently outperforms the S&P 500.

---
### Rebalance Logic:
- **Frequency**: Annual (12 Months).
- **Universe**: IT Services companies > $500M Revenue.
- **Selection**: Top 5 tickers by Atomic Stack Score.
- **Weighting**: Equal weight.

In [157]:
# --- PHASE 5: THE ATOMIC STRATEGY BACKTEST (CLEAN PIPELINE) ---

def run_phase5_backtest(tickers, start_date='2016-01-01', top_n=5):
    test_start = pd.to_datetime(start_date)
    today = pd.to_datetime(datetime.now().date())
    rebalance_dates = pd.date_range(start=test_start, end=today, freq='12MS')
    
    print(f"--- Starting Final Backtest Analysis (Local Data Only) ---")
    
    # 1. PRE-LOAD LOCAL PRICES (Absolute local truth)
    # Pivot includes SPY as we synced it in Phase 1.1
    all_prices = price_df.pivot(index='date', columns='ticker', values='adj_close')
    universe_prices = all_prices[all_prices.index >= test_start]
    
    # Check if SPY is available
    if 'SPY' not in all_prices.columns:
        print("CRITICAL: SPY benchmark missing from local pivot. Ensure Phase 1.1 ran successfully.")
        return

    spy_prices = all_prices['SPY']
    
    portfolio_value = 1.0
    history = pd.Series(index=universe_prices.index, dtype=float)
    history = history[history.index >= test_start]
    history.iloc[0] = 1.0
    
    last_picks_weights = {}

    for i, rb_date in enumerate(rebalance_dates):
        if rb_date >= today: break
        end_window = rebalance_dates[i+1] if i+1 < len(rebalance_dates) else today
        
        # 2. GENERATE ATOMIC DATA FOR THIS DATE
        period_data = []
        with np.errstate(divide='ignore', invalid='ignore'):
            for t in tickers:
                try:
                    rev = ttm_revenue(t, rb_date)
                    mkt = market_cap(t, rb_date)
                    if np.isnan(rev) or np.isnan(mkt) or rev <= 0: continue
                    
                    period_data.append({
                        'Ticker': t, 'Revenue_TTM': rev, 'Market_Cap': mkt,
                        'ROIC_%': ttm_roic(t, rb_date),
                        'F_Score_Factor': piotroski_f_score(t, rb_date),
                        'Growth_6Q_%': get_cumulative_growth(t, rb_date),
                        'PS_Current': mkt / rev,
                        'Rel_PS_Ind': (mkt/rev) / ind_medians.loc[tickers_meta.loc[t, 'zacks_x_ind_desc'], 'median_ps'],
                        'Rel_PS_Hist': (mkt/rev) / get_historical_ps_median(t, rb_date)
                    })
                except: continue
        
        atomic_df = pd.DataFrame(period_data).set_index('Ticker')
        
        # 3. APPLY CENTRALIZED OPPORTUNITY LOGIC
        opportunity_df = build_opportunity_matrix(atomic_df)
        if opportunity_df.empty: continue
        
        # 4. SELECTION & PROPORTIONAL WEIGHTING
        # "allocating tickers in proportion to their perceived tier or mosaic score"
        top_df = opportunity_df.head(top_n).copy()
        total_score = top_df['Stack_Score'].sum()
        top_df['Weight'] = top_df['Stack_Score'] / total_score
        
        picks = top_df.index.tolist()
        weights = top_df['Weight'].to_dict()
        last_picks_weights = weights
        print(f"[{rb_date.date()}] Selected: {picks} (Weights: {[f'{w:.2f}' for w in weights.values()]})")
        
        # 5. PORTFOLIO EXECUTION (Using Weighted Return)
        window = universe_prices.loc[(universe_prices.index >= rb_date) & (universe_prices.index <= end_window), picks]        
        window = window.ffill().bfill() 
                
        # Calculate daily returns for the window
        # Individual stock returns normalized to 1.0 at rb_date
        asset_returns = window / window.iloc[0]
        
        # Weighted portfolio return
        weighted_ret = (asset_returns * pd.Series(weights)).sum(axis=1)
        
        history.update(weighted_ret * portfolio_value)
        portfolio_value = history.loc[history.index <= end_window].iloc[-1]
    
    # 6. FINAL PERFORMANCE ANALYSIS
    history = history.ffill()
    spy_bench = (spy_prices / spy_prices.loc[spy_prices.index >= test_start].iloc[0]).reindex(history.index).ffill()
    
    total_ret = history.iloc[-1] - 1
    spy_ret = spy_bench.iloc[-1] - 1
    alpha = total_ret - spy_ret
    
    days = (history.index[-1] - history.index[0]).days
    cagr = ((history.iloc[-1]) ** (365.25 / days)) - 1
    
    # 7. RECENT TEAR ANALYSIS (Using local price data)
    five_months_ago = today - timedelta(days=150)
    current_picks = list(last_picks_weights.keys())
    recent_subset = all_prices.loc[all_prices.index >= five_months_ago, current_picks]
    recent_perf = (recent_subset.iloc[-1] / recent_subset.iloc[0] - 1) * 100

    print(f"\n--- Strategy Performance Metrics ---")
    print(f"Total Return: {total_ret*100:.2f}% (vs SPY: {spy_ret*100:.2f}%)")
    print(f"Alpha: {alpha*100:.2f}%")
    print(f"CAGR: {cagr*100:.2f}%")
    print(f"\n--- Recent Performer Analysis (Last 5 Months - Local Data) ---")
    print(f"Full Recent Selection Performance:\n{recent_perf.sort_values(ascending=False)}")

    # 8. VISUALIZATION
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=history.index, y=history, name='Strategy (Stack Weighted)'))
    fig.add_trace(go.Scatter(x=spy_bench.index, y=spy_bench, name='S&P 500 (SPY)', line=dict(dash='dot')))
    fig.update_layout(title=f"Atomic Strategy Output: Stack-Weighted Cluster (Alpha: {alpha*100:.1f}%)")
    fig.show()

run_phase5_backtest(it_services_universe)


--- Starting Final Backtest Analysis (Local Data Only) ---
[2016-01-01] Selected: ['CDW', 'SAIC', 'FJTSY', 'DOX', 'EPAM'] (Weights: ['0.30', '0.24', '0.18', '0.16', '0.12'])
[2017-01-01] Selected: ['CDW', 'DELL', 'FJTSY', 'EPAM', 'NTNX'] (Weights: ['0.24', '0.21', '0.19', '0.18', '0.17'])
[2018-01-01] Selected: ['DXC', 'DELL', 'CDW', 'SAIC', 'DOX'] (Weights: ['0.25', '0.22', '0.21', '0.18', '0.14'])
[2019-01-01] Selected: ['DXC', 'SAIC', 'CDW', 'DELL', 'EPAM'] (Weights: ['0.39', '0.18', '0.17', '0.15', '0.10'])
[2020-01-01] Selected: ['DELL', 'SAIC', 'CDW', 'DOX', 'DXC'] (Weights: ['0.27', '0.23', '0.19', '0.16', '0.15'])
[2021-01-01] Selected: ['SAIC', 'CDW', 'BZUN', 'DXC', 'DELL'] (Weights: ['0.27', '0.21', '0.20', '0.16', '0.16'])
[2022-01-01] Selected: ['BZUN', 'SAIC', 'CDW', 'VRT', 'DELL'] (Weights: ['0.38', '0.20', '0.15', '0.14', '0.13'])
[2023-01-01] Selected: ['TASK', 'UPST', 'DELL', 'CDW', 'DXC'] (Weights: ['0.31', '0.18', '0.17', '0.17', '0.16'])
[2024-01-01] Selected: ['TAS